# Frozen 100K crop sensitivity — inference only

Default: **P3 only**. Fill `MODEL_PATHS['P3']` with its existing `best.pth`, then edit the resource/image/source and frozen Val15K manifest paths. C/R7 are optional. No official CelebA full Test, training, detector or alignment is run.

Crop factors: **1.0 / 1.5 / 2.0 / 2.7**. Every image is decoded once per model/population to generate all four crops. Each crop receives its own source-Val15K threshold before LCC evaluation. Progress is printed from batch 1 and every 25 batches.

2.7x uses the original binary-v2 crop helper, including its scale limit and edge translation. Optional historical metrics can be supplied as a run-directory path or split-to-metric dictionary; see config comments. Without them, historical numerical reproduction is not claimed.

Primary measures are LCC evaluation/combined AUC. These checkpoints were **trained at 2.7x**; results measure inference-time sensitivity and do not automatically determine an optimal training or deployment crop. All four crops must be evaluated before interpreting deltas.


In [ ]:
# Mount the existing resources, frozen Val15K manifest, raw images and model source.
RESOURCE_ROOT='/kaggle/input/datasets/maithanhphuong/full-resources-v3/minifasnet_binary_crossdomain_resources_v2_lcc_pooled'
VAL15K_MANIFEST_PATH='/kaggle/input/datasets/maithanhphuong/100k-output/binary_crossdomain_scale100k_C_P3_Q3_v1/manifests/scale15k_val.csv'
CELEBA_ROOT='/kaggle/input/datasets/attentionlayer241/celeba-spoof-for-face-antispoofing/CelebA_Spoof_/CelebA_Spoof'
LCC_ROOT='/kaggle/input/datasets/faber24/lcc-fasd/LCC_FASD'
MINIFASNET_SOURCE_PATH='/kaggle/input/datasets/maithanhphuong/minifasnet/MiniFASNet.py'
MODEL_PATHS={'C':None,'P3':None,'R7':None}  # Fill only the selected best.pth paths.
MODEL_FILTER=['P3']
CROP_FACTORS=[1.0,1.5,2.0,2.7]
EXISTING_27X_METRICS={'C':None,'P3':None,'R7':None}
# Optional value: existing run directory, or {'val15k': metrics_dict_or_JSON_path,
# 'lcc_evaluation': metrics_dict_or_JSON_path, 'lcc_combined': ...}.
# A JSON file containing that split-to-metrics mapping is also accepted.
CONTROL_METRIC_ATOL=.005
OUTPUT_ROOT='/kaggle/working/crop_sensitivity_100k_v1'
BATCH_SIZE=256
NUM_WORKERS=4
AMP=True
LOG_EVERY_BATCHES=25
INPUT_SIZE=80
CROP_SCALE=2.7


In [ ]:
import importlib.util, json, math, time, zipfile
from pathlib import Path
import cv2, numpy as np, pandas as pd, torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, roc_curve
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from IPython.display import FileLink, display
print('[Setup] Preparing frozen crop-sensitivity inference...',flush=True)

PROTOCOL_TEXT='# Frozen 100K crop sensitivity, protocol v1\n\n## Question and hypothesis\n\nMeasure whether reducing face-crop context changes cross-domain separability on LCC for frozen C/P3/R7 checkpoints trained with 2.7x crops. Tighter crops may reduce nuisance/background shift; no outcome is assumed. This is inference-time sensitivity, not evidence of the performance of models trained with each alternative crop.\n\n## Paired controls\n\nSupport C100k_copy_pretrained, P3_100k_worst_ce25 and R7_100k_harmful_gated_worst. Default MODEL_FILTER is ["P3"]. Load existing best.pth only and run eval/no_grad with all parameters frozen. No training, optimizer, scheduler, BatchNorm update, detector, GT fallback, alignment, spectral perturbation, TTA or full CelebA Test is performed.\n\nEvaluate exactly [1.0, 1.5, 2.0, 2.7]. Reuse the binary-v2 MiniFASNetV2 wrapper and original parameterized CropImage-compatible helper. Its image-boundary behavior includes limiting context scale to fit the image and translating edge crops inward; do not replace it with a new center/clamp implementation. At 2.7 it must produce exactly the existing preprocessing. Geometry diagnostics use those same final pixel bounds. Boundary clipping records whether the requested centered context would exceed the image bounds or require the original helper\'s scale limit.\n\nKeep checkpoint, frozen SCRFD raw bbox, eligible candidates, images, 80x80 resize, BGR, float32 divided by 255, PAD score z_real-z_attack, and metric implementations fixed. No jitter/augmentation/normalization. Read the existing frozen scale15k_val.csv, never resample. Read the completed 00d LCC manifests/caches. Failed localization candidates remain unscored, with counts/coverage reported; image read errors stop rather than silently changing candidates.\n\n## Calibration and evaluation\n\nFor every model/crop pair infer Val15K, compute its minimum-ACER threshold with the existing three-class source implementation and freeze that threshold before LCC. Every LCC threshold-based metric uses only that pair\'s source threshold. No LCC threshold fitting or automatic crop selection.\n\nInfer physical LCC training/development/evaluation, raw counts 8299/2948/7580. Combined raw count is 18827. Concatenate prediction tables (including failed candidates) and recompute pooled metrics over scorable samples; never average split metrics. Primary outcomes are LCC evaluation/combined AUC. Secondary outcomes are EER, TPR@FPR1%, locked-source APCER/BPCER/HTER/ACER/accuracy. Also report source metrics, threshold, candidate/scored counts and coverage.\n\nThe 2.7x result is the control. Optional EXISTING_27X_METRICS supplies known source/LCC metric dictionaries, JSON files, or a run directory. Compare available metrics using absolute numerical tolerance 0.005 and exact candidate/scored counts. A discrepancy stops clearly. Without supplied references, report that historical numerical reproduction is unchecked; exact helper equivalence still holds. Do not invent reference values.\n\n## Efficiency and artifacts\n\nEach physical image is decoded once per enabled model/population; all four preprocessed crops are generated in that read, and model forwards run separately by crop within each batch. Only a bounded batch of crop tensors is held. No persistent full-image/tensor cache or large preflight framework. Print progress for source/LCC batches and artifact export.\n\nUse /kaggle/working/crop_sensitivity_100k_v1/<C|P3|R7>/crop_<s>/ for source/LCC predictions, metrics, freeze and crop diagnostics. For each model save comparison CSV/JSON, deltas versus 2.7, report and four plots. Multi-model tables/reports describe shared versus method-specific effects. Save protocol at root and per-model ZIPs; multiple models also produce the combined ZIP.\n\nDiagnostics are sample counts, fraction of requested crops limited by image boundaries and actual mean crop-width/image-width and crop-height/image-height, reported per population and pooled for LCC. They are descriptive, never selection criteria.\n\n## Interpretation and validation\n\nAn AUC change describes separability; HTER-only improvement may reflect calibration/operating-point effects. Joint AUC and EER/HTER improvement supports a context-shift hypothesis more strongly. Isolated small improvements warrant caution. Do not declare an optimal crop, make significance claims from a single checkpoint, or treat this as a matched train+eval study. Consistent improvements can motivate a later matched training experiment.\n\nBefore Kaggle, validate notebook parse/compile and cleared outputs, exact crop factors, P3-only filtering, original crop equivalence, eval-only calls, per-crop source calibration and pooled LCC computation with small offline checks. Full numerical reproduction and performance require actual Kaggle inference.\n'

def require_record(rec,key):
    if rec.get('status')!='VALID' or rec.get('usable_for_pad') is not True or rec.get('bbox_origin')!='SCRFD' or rec.get('source_key')!=key or not str(rec.get('bbox_source','')).startswith('SCRFD'):raise RuntimeError('Unscorable/non-SCRFD: '+key)
    box=rec.get('bbox_xyxy')
    if box is None or len(box)!=4 or 'crop_factor' in rec:raise RuntimeError('Not a raw bbox: '+key)
    x1,y1,x2,y2=map(float,box)
    if not np.isfinite(box).all() or x2<=x1 or y2<=y1:raise RuntimeError('Invalid bbox: '+key)
    return rec

def mini_crop_bgr(image,bbox_xyxy,scale=CROP_SCALE,size=INPUT_SIZE):
    # Exact 03b CropImage-compatible geometry.
    src_h,src_w=image.shape[:2];x1,y1,x2,y2=map(float,bbox_xyxy);box_w,box_h=x2-x1,y2-y1
    if box_w<=0 or box_h<=0 or src_w<2 or src_h<2:raise ValueError('Invalid crop geometry')
    used=min((src_h-1)/box_h,(src_w-1)/box_w,float(scale));new_w,new_h=box_w*used,box_h*used
    cx,cy=box_w/2+x1,box_h/2+y1;left,top=cx-new_w/2,cy-new_h/2;right,bottom=cx+new_w/2,cy+new_h/2
    if left<0:right-=left;left=0
    if top<0:bottom-=top;top=0
    if right>src_w-1:left-=right-src_w+1;right=src_w-1
    if bottom>src_h-1:top-=bottom-src_h+1;bottom=src_h-1
    lx,ty,rx,by=map(int,(left,top,right,bottom))
    if lx<0 or ty<0 or rx>=src_w or by>=src_h or rx<lx or by<ty:raise RuntimeError('Crop escaped image')
    patch=image[ty:by+1,lx:rx+1]
    if patch.size==0:raise RuntimeError('Empty crop')
    return cv2.resize(patch,(size,size))

def tensor_of(patch):
    if patch.shape!=(80,80,3) or patch.dtype!=np.uint8:raise ValueError('Expected BGR uint8 80x80')
    return torch.from_numpy(np.ascontiguousarray(patch.transpose(2,0,1))).float().div_(255.)

class PADNet(nn.Module):
    def __init__(self,copied):
        super().__init__();self.copied=copied
        self.trunk=module.MiniFASNetV2(embedding_size=128,conv6_kernel=(5,5),drop_p=.2,num_classes=3,img_channel=3)
        if copied:
            self.trunk.prob=nn.Identity()
            self.pad_head=nn.Linear(128,2);self.spoof_type_head=nn.Linear(128,11)
            self.lighting_head=nn.Linear(128,5);self.attributes_head=nn.Linear(128,40)
    def features(self,x):
        t=self.trunk
        for layer in (t.conv1,t.conv2_dw,t.conv_23,t.conv_3,t.conv_34,t.conv_4,t.conv_45,t.conv_5,t.conv_6_sep,t.conv_6_dw,t.conv_6_flatten):x=layer(x)
        if t.embedding_size!=512:x=t.linear(x)
        x=t.drop(t.bn(x))
        if x.ndim!=2 or x.shape[1]!=128:raise RuntimeError('MiniFAS representation is not [B,128]')
        return x
    def forward(self,x,aux=False):
        z=self.features(x)
        if not self.copied:return self.trunk.prob(z)
        pad=self.pad_head(z)
        if aux:return pad,self.spoof_type_head(z),self.lighting_head(z),self.attributes_head(z)
        return pad

def pad_score(logits):
    z=torch.as_tensor(logits)
    if z.shape[1]==3:return (z[:,0]-torch.logsumexp(z[:,1:],dim=1)).float().cpu().numpy()
    if z.shape[1]==2:return (z[:,0]-z[:,1]).float().cpu().numpy()
    raise ValueError('Expected two or three PAD logits')

def threshold_from_val(y3,scores):
    y=np.asarray(y3,dtype=np.int64);s=np.asarray(scores,dtype=np.float64)
    if len(y)!=len(s) or not len(y) or not np.isfinite(s).all() or set(np.unique(y))!={0,1,2}:raise ValueError('Val needs all three classes')
    order=np.argsort(s,kind='mergesort');v=s[order];real=(y[order]==0);nr=int(real.sum());na=len(y)-nr
    bound=np.r_[0,np.flatnonzero(np.diff(v))+1,len(v)];rp=np.r_[0,np.cumsum(real)];ap=np.r_[0,np.cumsum(~real)];pairs=[]
    for j,k in enumerate(bound):
        t=float(np.nextafter(v[0],-np.inf)) if j==0 else (float(np.nextafter(v[-1],np.inf)) if j==len(bound)-1 else float(v[k-1]/2+v[k]/2))
        pairs.append((.5*(rp[k]/nr+(na-ap[k])/na),abs(t),t))
    return min(pairs)[2]

def metrics(y3,scores,threshold,candidate_n):
    y=np.asarray(y3,dtype=int);s=np.asarray(scores,dtype=float);real=y==0
    if len(y)!=len(s) or set(real)!={False,True} or not np.isfinite(s).all():raise ValueError('Invalid metric population')
    fpr,tpr,thr=roc_curve(real.astype(int),s);eer_idx=int(np.argmin(abs(fpr-(1-tpr))))
    pred=s>=threshold;apcer=float(pred[~real].mean());bpcer=float((~pred[real]).mean())
    return {'auc':float(roc_auc_score(real.astype(int),s)),'eer':float(.5*(fpr[eer_idx]+1-tpr[eer_idx])),
      'tpr_at_fpr_1pct':float(np.interp(.01,fpr,tpr)),'apcer':apcer,'bpcer':bpcer,'acer':.5*(apcer+bpcer),
      'hter':.5*(apcer+bpcer),'binary_accuracy':float((pred==real).mean()),'locked_source_threshold':float(threshold),
      'candidate_n':int(candidate_n),'scored_n':len(y),'detector_coverage':len(y)/candidate_n}

def crop_geometry(image,bbox_xyxy,scale=CROP_SCALE,size=INPUT_SIZE):
    # Exact 03b CropImage-compatible geometry.
    src_h,src_w=image.shape[:2];x1,y1,x2,y2=map(float,bbox_xyxy);box_w,box_h=x2-x1,y2-y1
    if box_w<=0 or box_h<=0 or src_w<2 or src_h<2:raise ValueError('Invalid crop geometry')
    requested_w,requested_h=box_w*float(scale),box_h*float(scale)
    requested_cx,requested_cy=(x1+x2)/2,(y1+y2)/2
    limited=(requested_cx-requested_w/2<0 or requested_cy-requested_h/2<0 or requested_cx+requested_w/2>src_w-1 or requested_cy+requested_h/2>src_h-1)
    used=min((src_h-1)/box_h,(src_w-1)/box_w,float(scale));new_w,new_h=box_w*used,box_h*used
    cx,cy=box_w/2+x1,box_h/2+y1;left,top=cx-new_w/2,cy-new_h/2;right,bottom=cx+new_w/2,cy+new_h/2
    if left<0:right-=left;left=0
    if top<0:bottom-=top;top=0
    if right>src_w-1:left-=right-src_w+1;right=src_w-1
    if bottom>src_h-1:top-=bottom-src_h+1;bottom=src_h-1
    lx,ty,rx,by=map(int,(left,top,right,bottom))
    if lx<0 or ty<0 or rx>=src_w or by>=src_h or rx<lx or by<ty:raise RuntimeError('Crop escaped image')
    patch=image[ty:by+1,lx:rx+1]
    if patch.size==0:raise RuntimeError('Empty crop')
    return [float(limited or used<float(scale)),(rx-lx+1)/src_w,(by-ty+1)/src_h]

LOGICAL_NAMES={'C':'C100k_copy_pretrained','P3':'P3_100k_worst_ce25','R7':'R7_100k_harmful_gated_worst'}
SPLITS=('training','development','evaluation')
EXPECTED_LCC={'training':8299,'development':2948,'evaluation':7580,'combined':18827}
if not isinstance(MODEL_FILTER,(list,tuple)) or not MODEL_FILTER or len(set(MODEL_FILTER))!=len(MODEL_FILTER) or set(MODEL_FILTER)-set(LOGICAL_NAMES):
    raise ValueError('MODEL_FILTER must be a non-empty unique list of C, P3 or R7')
if CROP_FACTORS!=[1.0,1.5,2.0,2.7]:raise ValueError('Use exactly crop factors [1.0,1.5,2.0,2.7]')
if BATCH_SIZE<1 or NUM_WORKERS<0 or LOG_EVERY_BATCHES<1 or CONTROL_METRIC_ATOL<0:raise ValueError('Invalid execution configuration')
for name in MODEL_FILTER:
    if not MODEL_PATHS.get(name) or not Path(MODEL_PATHS[name]).is_file():raise FileNotFoundError('Set MODEL_PATHS['+repr(name)+'] to its existing best.pth')
if not torch.cuda.is_available():raise RuntimeError('Enable a Kaggle GPU for inference')
device=torch.device('cuda');use_amp=bool(AMP)
root=Path(RESOURCE_ROOT);out=Path(OUTPUT_ROOT)
source=Path(MINIFASNET_SOURCE_PATH)
if not source.is_file():raise FileNotFoundError(source)
spec=importlib.util.spec_from_file_location('frozen_crop_minifasnet',source)
module=importlib.util.module_from_spec(spec);spec.loader.exec_module(module)

def read_json(path):return json.loads(Path(path).read_text())
def write_json(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(value,indent=2,allow_nan=False)+'\n')

def load_model(name):
    payload=torch.load(MODEL_PATHS[name],map_location='cpu',weights_only=True)
    if isinstance(payload,dict) and 'state_dict' in payload:
        if payload.get('run_id') not in (None,LOGICAL_NAMES[name]):raise RuntimeError('Checkpoint run_id differs from selected model '+name)
        state=payload['state_dict']
    else:state=payload
    model=PADNet(True)
    model.load_state_dict(state,strict=True)
    model.requires_grad_(False);model.to(device);model.eval()
    return model

print('[Setup] Loading frozen Val15K and CelebA bbox cache...',flush=True)
val=pd.read_csv(VAL15K_MANIFEST_PATH,keep_default_na=False)
if len(val)!=15000 or val.sample_key.duplicated().any() or set(val.three_class_label)!={0,1,2} or not val.usable_for_pad.astype(bool).all():
    raise RuntimeError('Use the existing fully scorable frozen Val15K manifest')
celeba_records=read_json(root/'bbox/celeba_bbox_cache_full.json')['records']
lcc_frames={};lcc_records={}
for split in (*SPLITS,'combined'):
    manifest='lcc_official_evaluation_full' if split=='evaluation' else 'lcc_'+split+'_full'
    cache='lcc_official_evaluation_bbox_cache' if split=='evaluation' else 'lcc_'+split+'_bbox_cache'
    frame=pd.read_csv(root/'manifests'/(manifest+'.csv'),keep_default_na=False)
    if len(frame)!=EXPECTED_LCC[split] or frame.path.duplicated().any():raise RuntimeError('Unexpected LCC candidates: '+split)
    lcc_frames[split]=frame
    if split!='combined':
        print('[Setup] Loading LCC',split,'cache...',flush=True)
        recs=read_json(root/'bbox'/(cache+'.json'))['records']
        if set(frame.path)!=set(recs):raise RuntimeError('LCC cache/manifest keys differ: '+split)
        lcc_records[split]=recs
pooled_manifest=pd.concat([lcc_frames[s].assign(lcc_split=s) for s in SPLITS],ignore_index=True)
if not pooled_manifest.equals(lcc_frames['combined']):raise RuntimeError('LCC combined manifest differs from physical split concatenation')
out.mkdir(parents=True,exist_ok=True)
(out/'CROP_SENSITIVITY_100K_PROTOCOL_v1.md').write_text(PROTOCOL_TEXT)
print('[Setup] Ready. Selected models:',MODEL_FILTER,'GPU:',torch.cuda.get_device_name(0),flush=True)

class CropData(Dataset):
    def __init__(self,frame,split):
        self.split=split;usable=frame.loc[frame.usable_for_pad.astype(bool)]
        self.keys=usable['sample_key' if split=='val15k' else 'path'].astype(str).tolist()
        self.labels=usable['three_class_label' if split=='val15k' else 'label'].astype(int).tolist()
        self.recs=celeba_records if split=='val15k' else lcc_records[split]
        self.image_root=Path(CELEBA_ROOT) if split=='val15k' else (Path(LCC_ROOT)/'LCC_FASD_evaluation' if split=='evaluation' else Path(LCC_ROOT))
    def __len__(self):return len(self.keys)
    def __getitem__(self,i):
        key=self.keys[i];rec=require_record(self.recs[key],key)
        image=cv2.imread(str(self.image_root/key),cv2.IMREAD_COLOR)
        if image is None:raise FileNotFoundError(self.image_root/key)
        views=[];geometry=[]
        for factor in CROP_FACTORS:
            views.append(tensor_of(mini_crop_bgr(image,rec['bbox_xyxy'],scale=factor)))
            geometry.append(crop_geometry(image,rec['bbox_xyxy'],scale=factor))
        return torch.stack(views),self.labels[i],key,np.asarray(geometry,dtype=np.float64)

def loader(frame,split):
    return DataLoader(CropData(frame,split),batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,
                      pin_memory=True,drop_last=False)

def infer_crops(model,frame,split,name):
    dl=loader(frame,split);keys=[];ys=[];logits={s:[] for s in CROP_FACTORS};geometries=[];started=time.monotonic()
    print(f'[{name} {split}] Starting {len(dl.dataset)} scored images, {len(dl)} batches; all four crops',flush=True)
    model.eval()
    with torch.no_grad():
        for step,(views,y,k,g) in enumerate(dl,1):
            for index,factor in enumerate(CROP_FACTORS):
                x=views[:,index].to(device,non_blocking=True)
                with torch.cuda.amp.autocast(enabled=use_amp):z=model(x)
                logits[factor].append(z.float().cpu().numpy())
            keys.extend(k);ys.extend(y.numpy().tolist());geometries.append(g.numpy())
            if step==1 or step%LOG_EVERY_BATCHES==0 or step==len(dl):
                print(f'[{name} {split}] batch {step}/{len(dl)}, images {len(keys)}, elapsed {time.monotonic()-started:.1f}s',flush=True)
    expected=frame.loc[frame.usable_for_pad.astype(bool),'sample_key' if split=='val15k' else 'path'].astype(str).tolist()
    if keys!=expected or not geometries:raise RuntimeError('Scored candidates/order differ: '+split)
    return keys,np.asarray(ys,dtype=int),{s:np.concatenate(z) for s,z in logits.items()},np.concatenate(geometries)

def diagnostics(geometry):
    return {'number_of_evaluated_samples':len(geometry),'boundary_clipped_fraction':float(geometry[:,0].mean()),
      'mean_crop_width_over_image_width':float(geometry[:,1].mean()),
      'mean_crop_height_over_image_height':float(geometry[:,2].mean())}

def predictions(frame,keys,y,z,threshold,split):
    keycol='sample_key' if split=='val15k' else 'path';scores=pad_score(z)
    scored=pd.DataFrame({keycol:keys,'pad_logit_score':scores,'logit_0':z[:,0],'logit_1':z[:,1],
                        'pred_real':(scores>=threshold).astype(int)})
    result=frame.merge(scored,on=keycol,how='left',sort=False,validate='one_to_one')
    usable=frame.usable_for_pad.astype(bool)
    if result.loc[usable,'pad_logit_score'].isna().any() or result.loc[~usable,'pad_logit_score'].notna().any():raise RuntimeError('Prediction accounting differs')
    result['locked_threshold']=threshold
    return result

METRIC_KEYS=('auc','eer','tpr_at_fpr_1pct','apcer','bpcer','acer','hter','binary_accuracy')
def resolve_reference(value):
    if isinstance(value,dict):return value
    path=Path(value)
    if path.is_dir():
        result={}
        for split in ('val15k','lcc_training','lcc_development','lcc_evaluation','lcc_combined'):
            filenames=['val_metrics.json','val15k_metrics.json'] if split=='val15k' else [split+'_metrics.json']
            if split=='lcc_evaluation':filenames.append('lcc_metrics.json')
            file=next((path/f for f in filenames if (path/f).is_file()),None)
            if file:result[split]=read_json(file)
        if not result:raise ValueError('No reference metric files found in '+str(path))
        return result
    return read_json(path)

def check_control(name,results):
    value=EXISTING_27X_METRICS.get(name)
    if value is None:
        return {'status':'not_checked','reason':'No historical 2.7x metrics supplied; original crop implementation is reused.'}
    references=resolve_reference(value);checks=[]
    for split,reference in references.items():
        if split not in results:raise ValueError('Unknown reference split '+split+'; use val15k or lcc_<split>')
        old=reference if isinstance(reference,dict) else read_json(reference)
        new=results[split];compared=[]
        for key in ('candidate_n','scored_n'):
            if key in old and int(old[key])!=new[key]:raise RuntimeError(f'{name} 2.7x control candidate discrepancy: {split} {key}')
        for key in METRIC_KEYS:
            if key not in old:continue
            if not np.isfinite(float(old[key])) or abs(new[key]-float(old[key]))>CONTROL_METRIC_ATOL:
                raise RuntimeError(f'{name} 2.7x control mismatch: {split} {key}; new={new[key]}, reference={old[key]}, atol={CONTROL_METRIC_ATOL}. Check checkpoint/resources/preprocessing.')
            compared.append(key)
        if not compared:raise ValueError('Reference has no comparable metrics: '+split)
        checks.append({'split':split,'compared_metrics':compared})
    if not checks:raise ValueError('Empty historical 2.7x reference mapping')
    print('['+name+'] 2.7x historical control passed',flush=True)
    return {'status':'passed','absolute_tolerance':CONTROL_METRIC_ATOL,'checks':checks}


In [ ]:
SUMMARY_KEYS=('val15k_auc','val15k_acer','locked_threshold','lcc_evaluation_auc','lcc_evaluation_eer',
 'lcc_evaluation_hter','lcc_evaluation_tpr_at_fpr_1pct','lcc_combined_auc','lcc_combined_eer',
 'lcc_combined_hter','lcc_combined_tpr_at_fpr_1pct','boundary_clipped_fraction')
DELTA_KEYS=('lcc_evaluation_auc','lcc_combined_auc','lcc_combined_eer','lcc_combined_hter')
LIMITATION='These frozen models were trained using 2.7x preprocessing. This experiment measures inference-time crop sensitivity, not the performance of models trained consistently with each crop.'

def table(rows):
    lines=['| Model | Crop | '+' | '.join(SUMMARY_KEYS)+' |','|---|---:|'+'---:|'*len(SUMMARY_KEYS)]
    for r in rows:lines.append('| '+r['model']+' | '+str(r['crop_factor'])+' | '+' | '.join(f'{r[k]:.6f}' for k in SUMMARY_KEYS)+' |')
    return lines

def zip_model(name):
    folder=out/name;path=out.parent/f'crop_sensitivity_{name}_100k_v1.zip'
    print('[Export] Writing',path,flush=True)
    with zipfile.ZipFile(path,'w',compression=zipfile.ZIP_DEFLATED,allowZip64=True) as archive:
        for file in sorted(folder.rglob('*')):
            if file.is_file():archive.write(file,(Path(name)/file.relative_to(folder)).as_posix())
        archive.write(out/'CROP_SENSITIVITY_100K_PROTOCOL_v1.md','CROP_SENSITIVITY_100K_PROTOCOL_v1.md')
    display(FileLink(str(path)))

def summarize_model(name,pair_results,pair_diagnostics,control):
    folder=out/name;rows=[]
    for factor in CROP_FACTORS:
        metrics_by_split=pair_results[factor]
        row={'model':name,'logical_model':LOGICAL_NAMES[name],'crop_factor':factor,
          'locked_threshold':metrics_by_split['val15k']['locked_source_threshold'],
          'boundary_clipped_fraction':pair_diagnostics[factor]['lcc_combined']['boundary_clipped_fraction']}
        for split,mm in metrics_by_split.items():
            for key,value in mm.items():row[split+'_'+key]=value
        rows.append(row)
    baseline=next(r for r in rows if r['crop_factor']==2.7)
    for row in rows:
        for key in DELTA_KEYS:row['delta_vs_27_'+key]=row[key]-baseline[key]
    pd.DataFrame(rows).to_csv(folder/'crop_comparison.csv',index=False)
    write_json(folder/'crop_comparison.json',rows)
    report=['# '+name+' frozen 100K crop sensitivity','',LIMITATION,
      'Only crop scale changes. Each crop has its own locked CelebA-Val15K threshold; no LCC tuning.',
      'Primary endpoints: LCC Evaluation AUC and LCC Combined AUC. No automatic optimal crop is declared.','',
      *table(rows),'','## Deltas versus 2.7x','',
      '| Crop | Eval AUC delta | Combined AUC delta | Combined EER delta | Combined HTER delta |',
      '|---:|---:|---:|---:|---:|']
    for row in rows:report.append('| '+str(row['crop_factor'])+' | '+' | '.join(f'{row["delta_vs_27_"+key]:+.6f}' for key in DELTA_KEYS)+' |')
    report+=['','## Interpretation','']
    for row in rows:
        if row['crop_factor']==2.7:continue
        da=row['delta_vs_27_lcc_combined_auc'];de=row['delta_vs_27_lcc_combined_eer'];dh=row['delta_vs_27_lcc_combined_hter']
        if da>0 and de<0 and dh<0:
            interpretation='Pooled AUC, EER and HTER all improve descriptively; this pattern supports investigating excessive context in a later matched train+eval experiment.'
        elif da>0:
            interpretation='Pooled AUC improves descriptively; examine EER/HTER and official evaluation for trade-offs.'
        elif dh<0:
            interpretation='HTER improves without pooled AUC improvement; this may be an operating-point/calibration effect.'
        else:interpretation='No joint pooled AUC/HTER improvement versus the control.'
        report.append('- '+str(row['crop_factor'])+'x: '+interpretation)
    report+=['','Small or isolated differences do not establish a robust improvement. No uncertainty/significance claim is made.',
      'An AUC change indicates changed cross-domain separability; HTER alone can reflect calibration/operating-point effects.',
      'A consistent tighter-crop AUC improvement can motivate a later matched training/evaluation study, not immediate crop deployment.',
      'Historical 2.7x control status: '+control['status']+'.',
      'If references were not supplied, historical numerical reproduction remains unchecked.',
      'Boundary fraction in the summary is for pooled LCC. Per-population actual geometry diagnostics are saved in each crop_diagnostics.json.',
      'Candidate failures stay unscored for every factor. Counts/coverage and complete metrics are in the comparison files.']
    (folder/'crop_report.md').write_text('\n'.join(report)+'\n')
    fig,axes=plt.subplots(2,2,figsize=(10,7))
    for ax,key,title in zip(axes.flat,
       ('lcc_combined_auc','lcc_evaluation_auc','lcc_combined_eer','lcc_combined_hter'),
       ('LCC Combined AUC','LCC Evaluation AUC','LCC Combined EER','LCC Combined HTER')):
        ax.plot(CROP_FACTORS,[r[key] for r in rows],marker='o');ax.set_xticks(CROP_FACTORS)
        ax.set_xlabel('Crop factor');ax.set_ylabel(title);ax.grid(alpha=.25)
        ax.axvline(2.7,color='gray',linestyle='--',linewidth=.8)
    fig.suptitle(name+' frozen checkpoint crop sensitivity');fig.tight_layout()
    fig.savefig(folder/'crop_sensitivity.png',dpi=160);plt.close(fig)
    return rows

all_rows=[]
for name in MODEL_FILTER:
    print('['+name+'] Loading frozen checkpoint:',MODEL_PATHS[name],flush=True)
    folder=out/name;folder.mkdir(parents=True,exist_ok=True)
    write_json(folder/'config.json',{'model':name,'logical_model':LOGICAL_NAMES[name],
      'checkpoint':str(MODEL_PATHS[name]),'source':str(source),'resource_root':str(root),
      'val15k_manifest':VAL15K_MANIFEST_PATH,'celeba_root':CELEBA_ROOT,'lcc_root':LCC_ROOT,
      'crop_factors':CROP_FACTORS,'batch_size':BATCH_SIZE,'num_workers':NUM_WORKERS,'amp':use_amp,
      'input':'80x80 BGR float32 [0,1]','primary_metrics':['lcc_evaluation_auc','lcc_combined_auc']})
    model=load_model(name);pair_results={s:{} for s in CROP_FACTORS};pair_diagnostics={s:{} for s in CROP_FACTORS}
    thresholds={};physical_predictions={s:[] for s in CROP_FACTORS};physical_geometry=[]
    keys,y,z_by_crop,geometry=infer_crops(model,val,'val15k',name)
    for index,factor in enumerate(CROP_FACTORS):
        pair_folder=folder/f'crop_{factor:.1f}';pair_folder.mkdir(parents=True,exist_ok=True)
        z=z_by_crop[factor];locked=threshold_from_val(y,pad_score(z));thresholds[factor]=locked
        mm=metrics(y,pad_score(z),locked,len(val));pair_results[factor]['val15k']=mm
        pair_diagnostics[factor]['val15k']=diagnostics(geometry[:,index])
        write_json(pair_folder/'val15k_metrics.json',mm)
        predictions(val,keys,y,z,locked,'val15k').to_csv(pair_folder/'val15k_predictions.csv',index=False)
        freeze={'model':name,'logical_model':LOGICAL_NAMES[name],'checkpoint':str(MODEL_PATHS[name]),
          'crop_factor':factor,'locked_threshold':locked,'threshold_source':'frozen CelebA Val15K at this crop',
          'threshold_rule':'source minimum ACER, existing binary-v2 implementation','val15k_metrics':mm}
        write_json(pair_folder/'evaluation_freeze.json',freeze)
        print(f'[{name} crop {factor}] Val15K threshold frozen: {locked:.6f}; AUC {mm["auc"]:.6f}, ACER {mm["acer"]:.6f}',flush=True)
    del z_by_crop,geometry
    for split in SPLITS:
        frame=lcc_frames[split];keys,y,z_by_crop,geometry=infer_crops(model,frame,split,name)
        physical_geometry.append(geometry)
        for index,factor in enumerate(CROP_FACTORS):
            pair_folder=folder/f'crop_{factor:.1f}';z=z_by_crop[factor];locked=thresholds[factor]
            mm=metrics(y,pad_score(z),locked,len(frame));pair_results[factor]['lcc_'+split]=mm
            pair_diagnostics[factor]['lcc_'+split]=diagnostics(geometry[:,index])
            write_json(pair_folder/f'lcc_{split}_metrics.json',mm)
            pred=predictions(frame,keys,y,z,locked,split).assign(lcc_split=split)
            pred.to_csv(pair_folder/f'lcc_{split}_predictions.csv',index=False);physical_predictions[factor].append(pred)
        del z_by_crop,geometry
    all_geometry=np.concatenate(physical_geometry)
    for index,factor in enumerate(CROP_FACTORS):
        pair_folder=folder/f'crop_{factor:.1f}';pooled=pd.concat(physical_predictions[factor],ignore_index=True)
        columns=['path','label','status','usable_for_pad','bbox_origin','lcc_split']
        if not pooled[columns].equals(lcc_frames['combined'][columns]):raise RuntimeError('Combined candidate accounting differs')
        usable=pooled.usable_for_pad.astype(bool)
        mm=metrics(pooled.loc[usable,'label'],pooled.loc[usable,'pad_logit_score'],thresholds[factor],len(pooled))
        pair_results[factor]['lcc_combined']=mm
        pair_diagnostics[factor]['lcc_combined']=diagnostics(all_geometry[:,index])
        write_json(pair_folder/'lcc_combined_metrics.json',mm)
        pooled.to_csv(pair_folder/'lcc_combined_predictions.csv',index=False)
        write_json(pair_folder/'crop_diagnostics.json',{'crop_factor':factor,
          'boundary_policy':'requested centered context exceeds image bounds or original scale cap',
          'populations':pair_diagnostics[factor]})
    control=check_control(name,pair_results[2.7]);write_json(folder/'control_27x_check.json',control)
    all_rows.extend(summarize_model(name,pair_results,pair_diagnostics,control))
    del model,physical_predictions,physical_geometry,all_geometry
    torch.cuda.empty_cache();zip_model(name)
if len(MODEL_FILTER)>1:
    pd.DataFrame(all_rows).to_csv(out/'all_models_crop_comparison.csv',index=False)
    report=['# Multi-model frozen crop sensitivity','',LIMITATION,'',*table(all_rows),'',
      'Compare same-direction AUC deltas across C/P3/R7 for evidence of a shared preprocessing effect.',
      'Different delta directions or sizes can indicate method-specific interactions; single checkpoints cannot establish causation or significance.',
      'Missing/non-selected models are not inferred. No optimal crop is selected.','']
    for factor in CROP_FACTORS[:-1]:
        measured=[r for r in all_rows if r['crop_factor']==factor]
        report.append('- '+str(factor)+'x pooled AUC deltas: '+', '.join(r['model']+f' {r["delta_vs_27_lcc_combined_auc"]:+.6f}' for r in measured)+'.')
    (out/'all_models_crop_report.md').write_text('\n'.join(report)+'\n')
    archive_path=out.parent/'crop_sensitivity_100k_v1.zip'
    with zipfile.ZipFile(archive_path,'w',compression=zipfile.ZIP_DEFLATED,allowZip64=True) as archive:
        for file in sorted(out.rglob('*')):
            if file.is_file():archive.write(file,(Path(out.name)/file.relative_to(out)).as_posix())
    display(FileLink(str(archive_path)))
print('Crop sensitivity inference complete. Models:',MODEL_FILTER,flush=True)
